# Data Cleaning and Reprocessing

### Installing libraries and importing modules

In [154]:
# pip install required modules
!pip install pandas
!pip install openpyxl
!pip install fingertips_py

In [1]:
# basic setup - install pandas module
import pandas as pd
import numpy as np
import requests
import fingertips_py as ftp

ModuleNotFoundError: No module named 'fingertips_py'

### ONS - General Health 2011 / 2021

[General health by age, sex and deprivation, England and Wales: Census 2021](https://www.ons.gov.uk/peoplepopulationandcommunity/healthandsocialcare/healthandwellbeing/articles/generalhealthbyagesexanddeprivationenglandandwales/census2021?utm_source=chatgpt.com)

In [3]:
file_path = "../data/raw/ons_general_health.xlsx"
xls = pd.ExcelFile(file_path)
xls.sheet_names

['Cover_sheet',
 'Contents',
 'Notes',
 'Definitions',
 'Questions_asked',
 'Table 1',
 'Table 2',
 'Table 3',
 'Table 4',
 'Table 5',
 'Table 6',
 'Table 7',
 'Table 8',
 'Table 9',
 'Table 10',
 'Table 11',
 'Table 12']

In [4]:
# The Census dataset was loaded into a structured dataframe named census_health_sex to clearly distinguish it from other project datasets.
census_health_sex = pd.read_excel(file_path, sheet_name="Table 1", skiprows=4)

# The dataset did not begin at cell A1, so the header row was identified manually and the file was loaded by skipping the rows above the data table.

#### Exploring the data

In [5]:
census_health_sex.head()

,Year,Country,Area Code,Health Status,Sex,Count,Population,Age-standardised Percentage,Lower 95% Confidence Interval,Upper 95% Confidence Interval,Notes
0,2021,England,E92000001,Very good,Female,13653975,28833720,47.1,47.1,47.2,[z]
1,2021,England,E92000001,Very good,Male,13736855,27656330,47.9,47.9,48.0,[z]
2,2021,England,E92000001,Very good,Persons,27390815,56490055,47.5,47.5,47.5,[z]
3,2021,England,E92000001,Good,Female,9769630,28833720,34.2,34.1,34.2,[z]
4,2021,England,E92000001,Good,Male,9271115,27656330,34.2,34.1,34.2,[z]


In [6]:
census_health_sex.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Year                           60 non-null     int64  
 1   Country                        60 non-null     object 
 2   Area Code                      60 non-null     object 
 3   Health Status                  60 non-null     object 
 4   Sex                            60 non-null     object 
 5   Count                          60 non-null     int64  
 6   Population                     60 non-null     int64  
 7   Age-standardised Percentage    60 non-null     float64
 8   Lower 95% Confidence Interval  60 non-null     float64
 9   Upper 95% Confidence Interval  60 non-null     float64
 10  Notes                          60 non-null     object 
dtypes: float64(3), int64(3), object(5)
memory usage: 5.3+ KB


In [7]:
census_health_sex.describe()

,Year,Count,Population,Age-standardised Percentage,Lower 95% Confidence Interval,Upper 95% Confidence Interval
count,60.000000,6.000000e+01,6.000000e+01,60.000000,60.000000,60.000000
mean,2016.000000,3.855782e+06,1.927891e+07,20.000000,19.963333,20.038333
std,5.042195,6.357725e+06,1.968328e+07,17.291558,17.265376,17.310623
min,2011.000000,2.407000e+04,1.504235e+06,1.200000,1.200000,1.200000
25%,2011.000000,2.426112e+05,1.579678e+06,4.525000,4.525000,4.525000
50%,2016.000000,7.182375e+05,1.458832e+07,14.150000,14.150000,14.250000
75%,2021.000000,3.773086e+06,2.795068e+07,34.300000,34.200000,34.300000
max,2021.000000,2.739082e+07,5.649006e+07,47.900000,47.900000,48.000000


In [8]:
census_health_sex.shape

(60, 11)

#### Cleaning the data

**Step 1:** Standard column names:
- ensuring names are lower case,
- no symbols,
- no extra spaces.

In [9]:
census_health_sex.columns = (
    census_health_sex.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("%", "pct")
    .str.replace("-", "_")
)
census_health_sex.columns

Index(['year', 'country', 'area_code', 'health_status', 'sex', 'count',
       'population', 'age_standardised_percentage',
       'lower_95pct_confidence_interval', 'upper_95pct_confidence_interval',
       'notes'],
      dtype='object')

In [10]:
census_health_sex = census_health_sex.drop(columns=["notes", "area_code"])

In [11]:
num_cols = [
    "count",
    "population",
    "age_standardised_percentage",
    "lower_95pct_confidence_interval",
    "upper_95pct_confidence_interval"
]

for col in num_cols:
    census_health_sex[col] = pd.to_numeric(census_health_sex[col], errors="coerce")

In [12]:
# Rows for “Persons” were excluded as they represent combined male and female totals rather than a distinct group.

census_health_sex = census_health_sex[census_health_sex["sex"].isin(["Male", "Female"])]

#### Checking changes

In [13]:
census_health_sex.head()

,year,country,health_status,sex,count,population,age_standardised_percentage,lower_95pct_confidence_interval,upper_95pct_confidence_interval
0,2021,England,Very good,Female,13653975,28833720,47.1,47.1,47.2
1,2021,England,Very good,Male,13736855,27656330,47.9,47.9,48.0
3,2021,England,Good,Female,9769630,28833720,34.2,34.1,34.2
4,2021,England,Good,Male,9271115,27656330,34.2,34.1,34.2
6,2021,England,Fair,Female,3807185,28833720,13.2,13.2,13.2


In [14]:
census_health_sex.shape

(40, 9)

In [15]:
census_health_sex["sex"].value_counts()

sex
Female    20
Male      20
Name: count, dtype: int64

#### Saving to a file

In [16]:
cleaned_census_path = "../data/clean/cleaned_census_health_sex.csv"

census_health_sex.to_csv(cleaned_census_path, index=False)

#### Summary

- 40 rows of data
- 9 usable columns
- 20 male records / 20 female records
- Covers two time periods - 2011 and 2021

**Columns available:**
- year
- country
- health_status
- sex
- count
- population
- age_standardised_percentage
- lower_95pct_confidence_interval
- upper_95pct_confidence_interval

### Hospital admitted patient care 2023-24

[Hospital Admitted Patient Care Activity, 2023-24](https://digital.nhs.uk/data-and-information/publications/statistical/hospital-admitted-patient-care-activity/2023-24)

### Hospital admitted patient care 2023-24 - Patient Diagnosis

In [17]:
# Primary Diagnosis Summary table
diagnosis_summary_df = pd.read_excel(
    '../data/raw/hosp-epis-stat-admi-diag-2023-24-tab.xlsx', engine="openpyxl", sheet_name='Primary Diagnosis Summary')

diagnosis_summary_df.sample(5)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44
103,I20-I25,Ischaemic heart diseases,364343,225279,244381,116478,3484,133635,72659,4191,...,49503,52962,36290,24000,12167,59496,955286,18913,1961,4721
216,T80-T88,Complications of surgical & medical care nec.,304127,237207,164045,137931,2151,143377,74356,16003,...,28778,33599,25599,19150,10904,58521,978175,53451,4781,827
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Zero bed day cases §,NaN,NaN
50,D55-D59,Haemolytic anaemias,63496,56389,31112,31979,405,15232,20291,20655,...,1022,1229,881,658,342,39411,74195,4131,550,84
1,Primary diagnosis: summary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# Exploring data shape & types
print(diagnosis_summary_df.shape)
print(diagnosis_summary_df.info())

(234, 45)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234 entries, 0 to 233
Data columns (total 45 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   227 non-null    object
 1   Unnamed: 1   216 non-null    object
 2   Unnamed: 2   218 non-null    object
 3   Unnamed: 3   218 non-null    object
 4   Unnamed: 4   218 non-null    object
 5   Unnamed: 5   218 non-null    object
 6   Unnamed: 6   218 non-null    object
 7   Unnamed: 7   218 non-null    object
 8   Unnamed: 8   218 non-null    object
 9   Unnamed: 9   218 non-null    object
 10  Unnamed: 10  218 non-null    object
 11  Unnamed: 11  213 non-null    object
 12  Unnamed: 12  213 non-null    object
 13  Unnamed: 13  218 non-null    object
 14  Unnamed: 14  218 non-null    object
 15  Unnamed: 15  217 non-null    object
 16  Unnamed: 16  218 non-null    object
 17  Unnamed: 17  218 non-null    object
 18  Unnamed: 18  218 non-null    object
 19  Unnamed: 19  218 no

In [19]:
# Show top 12 rows to see where column headers & data starts
diagnosis_summary_df.head(12)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Primary diagnosis: summary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,The summary table groups together broadly asso...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Copyright © 2024, NHS England.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Activity in English NHS Hospitals and English ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,For more information on data quality please se...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Zero bed day cases §,NaN,NaN
8,Primary diagnosis: summary code and description,NaN,Finished consultant episodes,Finished Admission Episodes,Male \n(FCE),Female \n(FCE),Gender Unknown \n(FCE),Emergency \n(FAE),Waiting list \n(FAE),Planned (FAE),...,Age 70-74 \n(FCE),Age 75-79 \n(FCE),Age 80-84 \n(FCE),Age 85-89 \n(FCE),Age 90+ \n(FCE),Day case \n(FCE),FCE bed days,Emergency \n(FAE),Elective\n(FAE),Other\n(FAE)
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# applying column names using values in row index 8
diag_columns = diagnosis_summary_df.loc[8, :].values.flatten().tolist()

# First and second indices are from a merged cell so there is a nan at list index 1, replacing these manually:
diag_columns[0:2] = ["Code","Description"]

diagnosis_summary_df.columns = diag_columns
diagnosis_summary_df.head()

,Code,Description,Finished consultant episodes,Finished Admission Episodes,Male \n(FCE),Female \n(FCE),Gender Unknown \n(FCE),Emergency \n(FAE),Waiting list \n(FAE),Planned (FAE),...,Age 70-74 \n(FCE),Age 75-79 \n(FCE),Age 80-84 \n(FCE),Age 85-89 \n(FCE),Age 90+ \n(FCE),Day case \n(FCE),FCE bed days,Emergency \n(FAE),Elective\n(FAE),Other\n(FAE)
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Primary diagnosis: summary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,The summary table groups together broadly asso...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Copyright © 2024, NHS England.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
diag_nulls = diagnosis_summary_df.isnull().sum()
diag_nulls

# most columns have 16 nulls - the consistency of this total is due to the formatting of the spreadsheet having blank rows

Code                               7
Description                       18
Finished consultant episodes      16
Finished Admission Episodes       16
Male \n(FCE)                      16
Female \n(FCE)                    16
Gender Unknown \n(FCE)            16
Emergency \n(FAE)                 16
Waiting list \n(FAE)              16
Planned (FAE)                     16
Other (FAE)                       16
Mean time waited \n(Days)         21
Median time waited \n(Days)       21
Mean length of stay \n(Days)      16
Median length of stay \n(Days)    16
Mean age \n(Years)                17
Age 0 \n(FCE)                     16
Age 1-4 \n(FCE)                   16
Age 5-9 \n(FCE)                   16
Age 10-14 \n(FCE)                 16
Age 15 \n(FCE)                    16
Age 16 \n(FCE)                    16
Age 17 \n(FCE)                    16
Age 18 \n(FCE)                    16
Age 19 \n(FCE)                    16
Age 20-24 \n(FCE)                 16
Age 25-29 \n(FCE)                 16
A

In [23]:
# Cleaning column names
def clean_table(dataframe) :
  dataframe.columns = (dataframe.columns
                  .str.strip()
                  .str.lower()
                  .str.replace('\n', '')
                  .str.replace(' ', '_'))
  # removing rows with all nans in primary diagnosis column, as this column is complete due to the data aggregation
  dataframe = dataframe.dropna(how='all', subset=['description'])

  return dataframe

diag_summary_clean = clean_table(diagnosis_summary_df)
diag_summary_clean.head()

,code,description,finished_consultant_episodes,finished_admission_episodes,male_(fce),female_(fce),gender_unknown_(fce),emergency_(fae),waiting_list_(fae),planned_(fae),...,age_70-74_(fce),age_75-79_(fce),age_80-84_(fce),age_85-89_(fce),age_90+_(fce),day_case_(fce),fce_bed_days,emergency_(fae),elective(fae),other(fae)
11,A00-A09,Intestinal infectious diseases,203152,142555,87268,114810,1074,114231,24946,2782,...,14269,17504,14927,12774,8248,26163,425201,36569,420,179
12,A15-A19,Tuberculosis,6134,2600,3924,2186,24,1634,712,184,...,320,194,138,69,26,719,43028,198,21,4
13,A20-A28,Certain zoonotic bacterial diseases,361,201,198,163,0,146,48,3,...,21,11,18,22,7,46,1411,49,2,0
14,A30-A49,Other bacterial diseases,266344,121395,144404,121099,841,114125,3180,2953,...,29839,39743,35696,32859,24037,3114,1413520,5204,150,70
15,A50-A64,Infections with a predominantly sexual mode of...,2823,2475,1388,1416,19,997,1245,207,...,0,0,0,0,0,1286,4482,378,53,5


In [24]:
# function to convert to long format
def long_format(dataframe):
  melted_df = pd.melt(
      dataframe,
      id_vars=['code', 'description'],
      value_vars=['male_(fce)',	'female_(fce)', 'gender_unknown_(fce)'],
      var_name='sex',
      value_name='diagnosis_count')

  # replace values to improve readability
  melted_df['sex'] = melted_df['sex'].replace({
      'male_(fce)': 'Male',
      'female_(fce)': 'Female',
      'gender_unknown_(fce)': 'Unknown'
  })

  # Convert 'diagnosis_count' to numeric
  melted_df['diagnosis_count'] = pd.to_numeric(melted_df['diagnosis_count'], errors='coerce')

  return melted_df

melt_diag_summary = long_format(diag_summary_clean)
melt_diag_summary.sample(5)

,code,description,sex,diagnosis_count
20,C00-C14,Malignant neoplasm of lip-oral cavity and pharynx,Male,21732
565,N00-N08,Glomerular diseases,Unknown,229
174,R10-R19,Symptoms & signs inv. the digestive system & a...,Male,197018
85,H60-H62,Diseases of external ear,Male,7656
362,O20-O29,Other maternal disorders predominantly related...,Female,263638


In [25]:
# Primary Diagnosis 3 Character
# This sheet offers a more granular version of the data
diagnosis_3_char_df = pd.read_excel(
    '../data/raw/hosp-epis-stat-admi-diag-2023-24-tab.xlsx', engine="openpyxl", sheet_name='Primary Diagnosis 3 Character')

# diagnosis_3_char_df.head(15) # column headers are at row index 9 for this sheet and there are also 5 blank columns after description
diagnosis_3_char_df = diagnosis_3_char_df.drop(diagnosis_3_char_df.columns[2:7], axis=1)

diag_3_char_columns = diagnosis_3_char_df.loc[9, :].values.flatten().tolist()

diag_3_char_columns[0:2] = ["Code","Description"]
# diag_3_char_columns
diagnosis_3_char_df.columns = diag_3_char_columns

diag_3_char_clean = clean_table(diagnosis_3_char_df)

melt_diag_3_char = long_format(diag_3_char_clean)
melt_diag_3_char.sample(5)

,code,description,sex,diagnosis_count
3912,J00,Acute nasopharyngitis [common cold],Unknown,12.0
2843,Q61,Cystic kidney disease,Female,637.0
637,I37,Pulmonary valve disorders,Male,246.0
4522,R32,Unspecified urinary incontinence,Unknown,64.0
94,B22,Human immunodeficiency virus [HIV] disease res...,Male,110.0


In [26]:
# checking how many 'unknown's there are in gender column
diag_count_df = melt_diag_3_char.groupby('sex')['diagnosis_count'].sum().to_frame()
print(diag_count_df)

         diagnosis_count
sex                     
Female        11687625.0
Male           9609508.0
Unknown         153660.0


In [27]:
# removing Unknown category by saving subset, as it makes up a small part of totals
melt_diag_summary = melt_diag_summary[melt_diag_summary['sex'] != 'Unknown'].copy()
melt_diag_summary.head()

,code,description,sex,diagnosis_count
0,A00-A09,Intestinal infectious diseases,Male,87268
1,A15-A19,Tuberculosis,Male,3924
2,A20-A28,Certain zoonotic bacterial diseases,Male,198
3,A30-A49,Other bacterial diseases,Male,144404
4,A50-A64,Infections with a predominantly sexual mode of...,Male,1388


In [28]:
melt_diag_3_char = melt_diag_3_char[melt_diag_3_char['sex'] != 'Unknown'].copy()
melt_diag_3_char.head()

,code,description,sex,diagnosis_count
0,A00,Cholera,Male,9.0
1,A01,Typhoid and paratyphoid fevers,Male,552.0
2,A02,Other salmonella infections,Male,1014.0
3,A03,Shigellosis,Male,424.0
4,A04,Other bacterial intestinal infections,Male,13214.0


#### Diagnosis 3 Character Code Categories

[NHS Classifications Browser](https://classbrowser.nhs.uk/#/book/ICD-10-5TH-Edition/vol1/volume1threeCode.htm+icd10firstObjec)

In [29]:
#assinginhg categories manually as some match 1st character, some match 1st and 2nd
categories = {
    'A': 'Certain infectious and parasitic diseases (A00–B99)',
    'B': 'Certain infectious and parasitic diseases (A00–B99)',
    'C': 'Neoplasms (C00–D48)',
    'D0': 'Neoplasms (C00–D48)',
    'D1': 'Neoplasms (C00–D48)',
    'D2': 'Neoplasms (C00–D48)',
    'D3': 'Neoplasms (C00–D48)',
    'D4': 'Neoplasms (C00–D48)',
    'D5': 'Diseases of the blood and blood-forming organs (D50–D89)',
    'D6': 'Diseases of the blood and blood-forming organs (D50–D89)',
    'D7': 'Diseases of the blood and blood-forming organs (D50–D89)',
    'D8': 'Diseases of the blood and blood-forming organs (D50–D89)',
    'E': 'Endocrine, nutritional and metabolic diseases (E00–E90)',
    'F': 'Mental and behavioural disorders (F00–F99)',
    'G': 'Diseases of the nervous (G00–G99)',
    'H0': 'Diseases of the eye and adnexa (H00–H59)',
    'H1': 'Diseases of the eye and adnexa (H00–H59)',
    'H2': 'Diseases of the eye and adnexa (H00–H59)',
    'H3': 'Diseases of the eye and adnexa (H00–H59)',
    'H4': 'Diseases of the eye and adnexa (H00–H59)',
    'H5': 'Diseases of the eye and adnexa (H00–H59)',
    'H6': 'Diseases of the ear and mastoid process (H60–H95)',
    'H7': 'Diseases of the ear and mastoid process (H60–H95)',
    'H8': 'Diseases of the ear and mastoid process (H60–H95)',
    'H9': 'Diseases of the ear and mastoid process (H60–H95)',
    'I': 'Diseases of the circulatory system (I00–I99)',
    'J': 'Diseases of the respiratory system (J00–J99)',
    'K': 'Diseases of the digestive system (K00–K93)',
    'L': 'Diseases of the skin and subcutaneous tissue (L00–L99)',
    'M': 'Diseases of the musculoskeletal system and connective tissue (M00–M99)',
    'N': 'Diseases of the genitourinary system (N00–N99)',
    'O': 'Pregnancy, childbirth and the puerperium (O00–O99)',
    'P': 'Certain conditions originating in the perinatal period (P00–P96)',
    'Q': 'Congenital malformations, deformations and chromosomal abnormalities (Q00–Q99)',
    'R': 'Symptoms, signs and abnormal clinical and laboratory findings (R00–R99)',
    'S': 'Injury, poisoning and certain other consequences of external causes (S00–T98)',
    'T': 'Injury, poisoning and certain other consequences of external causes (S00–T98)',
    'V': 'External causes of morbidity and mortality (V01–Y98)',
    'W': 'External causes of morbidity and mortality (V01–Y98)',
    'X': 'External causes of morbidity and mortality (V01–Y98)',
    'Y': 'External causes of morbidity and mortality (V01–Y98)',
    'Z': 'Factors influencing health status and contact with health services (Z00–Z99)',
    'U': 'Codes for special purposes (U00–U85)'
}

melt_diag_summary['category'] = melt_diag_summary.loc[:, 'category'] = np.select(
    [melt_diag_summary['code'].str.startswith(cat) for cat in categories.keys()],
    list(categories.values()),
    default='Other'
)

display(melt_diag_summary)

,code,description,sex,diagnosis_count,category
0,A00-A09,Intestinal infectious diseases,Male,87268,Certain infectious and parasitic diseases (A00...
1,A15-A19,Tuberculosis,Male,3924,Certain infectious and parasitic diseases (A00...
2,A20-A28,Certain zoonotic bacterial diseases,Male,198,Certain infectious and parasitic diseases (A00...
3,A30-A49,Other bacterial diseases,Male,144404,Certain infectious and parasitic diseases (A00...
4,A50-A64,Infections with a predominantly sexual mode of...,Male,1388,Certain infectious and parasitic diseases (A00...
...,...,...,...,...,...
427,Z30-Z39,Health services in circumstances related to re...,Female,316606,Factors influencing health status and contact ...
428,Z40-Z54,Persons encountering health services for speci...,Female,51752,Factors influencing health status and contact ...
429,Z55-Z65,Potential health hazards reltd. to socioeconom...,Female,737,Factors influencing health status and contact ...
430,Z70-Z76,Persons encountering health services in other ...,Female,12038,Factors influencing health status and contact ...


In [30]:
melt_diag_3_char['category'] = melt_diag_3_char.loc[:, 'category'] = np.select(
    [melt_diag_3_char['code'].str.startswith(cat) for cat in categories.keys()],
    list(categories.values()),
    default='Other'
)

display(melt_diag_3_char)

,code,description,sex,diagnosis_count,category
0,A00,Cholera,Male,9.0,Certain infectious and parasitic diseases (A00...
1,A01,Typhoid and paratyphoid fevers,Male,552.0,Certain infectious and parasitic diseases (A00...
2,A02,Other salmonella infections,Male,1014.0,Certain infectious and parasitic diseases (A00...
3,A03,Shigellosis,Male,424.0,Certain infectious and parasitic diseases (A00...
4,A04,Other bacterial intestinal infections,Male,13214.0,Certain infectious and parasitic diseases (A00...
...,...,...,...,...,...
3223,Z91,"Personal history of risk-factors, not elsewher...",Female,25.0,Factors influencing health status and contact ...
3224,Z92,Personal history of medical treatment,Female,2.0,Factors influencing health status and contact ...
3225,Z94,Transplanted organ and tissue status,Female,7.0,Factors influencing health status and contact ...
3226,Z95,Presence of cardiac and vascular implants and ...,Female,0.0,Factors influencing health status and contact ...


In [31]:
melt_diag_summary.to_csv('diagnosis_summary_clean.csv', index=False)

In [78]:
melt_diag_3_char.to_csv('diagnosis_3_char_clean.csv', index=False)

### Hospital admitted patient care 2023-24 - Procedures

In [33]:
# Procedures Summary table
file_path = '../data/raw/hosp-epis-stat-admi-proc-2023-24-tab-v2.xlsx'

procedures_df = pd.read_excel(
    file_path,
    engine='openpyxl',
    sheet_name='Primary Procedure Summary'
)

procedures_df.head(15)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Main procedures and interventions: summary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,The summary table groups together broadly asso...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Copyright © 2024, NHS England.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Activity in English NHS Hospitals and English ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,For more information on data quality please se...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Zero bed day cases §,NaN,NaN
8,Main procedures and interventions: summary cod...,NaN,NaN,NaN,NaN,NaN,Finished Consultant Episodes,Finished Admission Episodes,Male (FCE),Female (FCE),...,Age 70-74 (FCE),Age 75-79 (FCE),Age 80-84 (FCE),Age 85-89 (FCE),Age 90+ (FCE),Day case (FCE),FCE bed days (FCE),Emergency (FAE),Elective (FAE),Other (FAE)
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
# drooping rows with all nans
procedures_df = procedures_df.dropna(how='all')

In [38]:
file_path = '../data/raw/hosp-epis-stat-admi-proc-2023-24-tab-v2.xlsx'

procedures_df = pd.read_excel(
    file_path,
    engine='openpyxl',
    sheet_name='Primary Procedure Summary',
    skiprows=9,

)
procedures_df.head(10)

,Main procedures and interventions: summary code and description,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Finished Consultant Episodes,Finished Admission Episodes,Male (FCE),Female (FCE),...,Age 70-74 (FCE),Age 75-79 (FCE),Age 80-84 (FCE),Age 85-89 (FCE),Age 90+ (FCE),Day case (FCE),FCE bed days (FCE),Emergency (FAE).1,Elective (FAE),Other (FAE)
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,All Procedures,NaN,NaN,NaN,NaN,NaN,21450875.0,17560128.0,9609513.0,11687625.0,...,1854892.0,2183072.0,1692024.0,1272957.0,790982.0,7959024.0,48324062.0,2419256.0,196436.0,630653.0
2,A Nervous System (A01-A84),NaN,NaN,NaN,NaN,NaN,307140.0,282713.0,130853.0,174631.0,...,22355.0,24340.0,15499.0,8930.0,3179.0,202683.0,678994.0,6607.0,3516.0,488.0
3,A1 Tissue of brain (A01-A11),NaN,NaN,NaN,NaN,NaN,9898.0,8901.0,5491.0,4382.0,...,978.0,695.0,159.0,32.0,5.0,1318.0,72569.0,24.0,198.0,0.0
4,A2 Ventricle of brain and subarachnoid space (...,NaN,NaN,NaN,NaN,NaN,7404.0,5872.0,3706.0,3695.0,...,402.0,514.0,222.0,73.0,3.0,1044.0,71835.0,73.0,41.0,1.0
5,A3 Cranial nerves (A24-A36),NaN,NaN,NaN,NaN,NaN,3325.0,3250.0,1462.0,1851.0,...,177.0,144.0,73.0,46.0,18.0,1594.0,7601.0,21.0,279.0,0.0
6,A4 Meninges of brain (A38-A43),NaN,NaN,NaN,NaN,NaN,6204.0,5309.0,3541.0,2663.0,...,655.0,848.0,578.0,419.0,158.0,17.0,63466.0,4.0,7.0,0.0
7,A5 Spinal cord and other contents of spinal ca...,NaN,NaN,NaN,NaN,NaN,141834.0,127748.0,61877.0,79476.0,...,9128.0,9417.0,5523.0,2766.0,731.0,87254.0,305413.0,4929.0,1743.0,300.0
8,A6 Peripheral nerves (A59-A73),NaN,NaN,NaN,NaN,NaN,116435.0,114265.0,43894.0,72041.0,...,10038.0,11726.0,8289.0,5132.0,2052.0,105773.0,45202.0,1293.0,1116.0,155.0
9,A7 Other parts of nervous system (A75-A84),NaN,NaN,NaN,NaN,NaN,22040.0,17368.0,10882.0,10523.0,...,977.0,996.0,655.0,462.0,212.0,5683.0,112908.0,263.0,132.0,32.0


In [40]:
procedures_df.columns

Index(['Main procedures and interventions: summary code and description',
       'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5',
       'Finished Consultant Episodes', 'Finished Admission Episodes',
       'Male (FCE)', 'Female (FCE)', 'Gender Unknown (FCE)', 'Emergency (FAE)',
       'Waiting list (FAE)', 'Planned (FAE)', 'Other Admission Method (FAE)',
       'Mean time waited (Days)', 'Median time waited (Days)',
       'Mean length of stay (Days)', 'Median length of stay (Days)',
       'Mean age (Years)', 'Age 0 (FCE)', 'Age 1-4 (FCE)', 'Age 5-9 (FCE)',
       'Age 10-14 (FCE)', 'Age 15 (FCE)', 'Age 16 (FCE)', 'Age 17 (FCE)',
       'Age 18 (FCE)', 'Age 19 (FCE)', 'Age 20-24 (FCE)', 'Age 25-29 (FCE)',
       'Age 30-34 (FCE)', 'Age 35-39 (FCE)', 'Age 40-44 (FCE)',
       'Age 45-49 (FCE)', 'Age 50-54 (FCE)', 'Age 55-59 (FCE)',
       'Age 60-64 (FCE)', 'Age 65-69 (FCE)', 'Age 70-74 (FCE)',
       'Age 75-79 (FCE)', 'Age 80-84 (FCE)', 'Age 85-89 (FCE)',
      

In [41]:
procedures_df = procedures_df.dropna(how='all')

In [42]:
procedures_df = procedures_df.rename(columns={
    'Main procedures and interventions: summary code and description': 'procedure_group',
    'Male (FCE)': 'male_fce',
    'Female (FCE)': 'female_fce'
})

procedures_df[['procedure_group', 'male_fce', 'female_fce']].head()

,procedure_group,male_fce,female_fce
1,All Procedures,9609513.0,11687625.0
2,A Nervous System (A01-A84),130853.0,174631.0
3,A1 Tissue of brain (A01-A11),5491.0,4382.0
4,A2 Ventricle of brain and subarachnoid space (...,3706.0,3695.0
5,A3 Cranial nerves (A24-A36),1462.0,1851.0


In [43]:
procedures_df = procedures_df[[
    'procedure_group',
    'male_fce',
    'female_fce'
]]

In [44]:
# Remove rows where male and female counts are missing
procedures_df = procedures_df.dropna(subset=['male_fce', 'female_fce'], how='all')

In [45]:
for col in ['male_fce', 'female_fce']:
    procedures_df[col] = (
        procedures_df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .replace('nan', pd.NA)
    )


    procedures_df[col] = pd.to_numeric(procedures_df[col], errors='coerce')
    procedures_df[col] = procedures_df[col].round(0).astype('Int64')

In [46]:
# Converting to long format
tidy_proc = procedures_df.melt(
    id_vars=['procedure_group'],
    value_vars=['male_fce', 'female_fce'],
    var_name='sex',
    value_name='procedure_count')


tidy_proc['sex'] = tidy_proc['sex'].map({
    'male_fce': 'Male',
    'female_fce': 'Female'})

# Drop rows
tidy_proc = tidy_proc.dropna(subset=['procedure_count'])

tidy_proc.tail()

,procedure_group,sex,procedure_count
241,"W4 Overflow other bones and joints (O06-O10, O...",Female,12421
242,X Miscellaneous Operations (X01-X97),Female,1372640
243,X1 Operations covering multiple systems (X01-X27),Female,7429
244,X2 Miscellaneous operations (X28-X69),Female,888960
245,X3 Specified Drug therapy (X70-X98),Female,476251


In [47]:
# Dropping rows where male/female values are missing
procedures_df = procedures_df.dropna(subset=['male_fce', 'female_fce'], how='all')

In [90]:
procedures_df.to_csv('procedures_primary_clean.csv', index=False)

In [48]:
# exploring the resulting cleaned data
print(procedures_df.shape)
print(procedures_df.head())

(123, 3)
                                     procedure_group  male_fce  female_fce
1                                     All Procedures   9609513    11687625
2                         A Nervous System (A01-A84)    130853      174631
3                       A1 Tissue of brain (A01-A11)      5491        4382
4  A2 Ventricle of brain and subarachnoid space (...      3706        3695
5                        A3 Cranial nerves (A24-A36)      1462        1851


In [49]:
# adding procedure groups to dataframe
procedures_df[procedures_df['procedure_group'].str.match(r'^[A-Z]\s', na=False)]['procedure_group']

2                             A Nervous System (A01-A84)
10                 B Endocrine System & Breast (B01-B41)
15                                       C Eye (C01-C91)
24                                       D Ear (D01-D28)
28                         E Respiratory Tract (E01-E98)
36                                     F Mouth (F01-F63)
43                     G Upper Digestive Tract (G01-G82)
49                     H Lower Digestive Tract (H01-H71)
54     J Other Abdominal Organs - Principally Digesti...
61                                     K Heart (K01-K78)
66       L Arteries and Veins (L01- L99,O01-O05,O15,O20)
75                                   M Urinary (M01-M86)
81                       N Male Genital Organs (N01-N35)
85                P Lower Female Genital Tract (P01-P32)
88                Q Upper Female Genital Tract (Q01-Q58)
92     R Female Genital Tract Associated With Pregnan...
97                                      S Skin (S01-S70)
100                            

In [50]:
subheading_mask = procedures_df['procedure_group'].str.match(r'^[A-Z]\s', na=False)

In [51]:
procedures_df.loc[subheading_mask, 'procedure_group'].unique()

array(['A Nervous System (A01-A84)',
       'B Endocrine System & Breast (B01-B41)', 'C Eye (C01-C91)',
       'D Ear (D01-D28)', 'E Respiratory Tract (E01-E98)',
       'F Mouth (F01-F63)', 'G Upper Digestive Tract (G01-G82)',
       'H Lower Digestive Tract (H01-H71)',
       'J Other Abdominal Organs - Principally Digestive (J01-J77)',
       'K Heart (K01-K78)',
       'L Arteries and Veins (L01- L99,O01-O05,O15,O20)',
       'M Urinary (M01-M86)', 'N Male Genital Organs (N01-N35)',
       'P Lower Female Genital Tract (P01-P32)',
       'Q Upper Female Genital Tract (Q01-Q58)',
       'R Female Genital Tract Associated With Pregnancy Childbirth & Puerperium (R01-R43)',
       'S Skin (S01-S70)', 'T Soft Tissue (T01-T99)',
       'U Diagnostic Testing & Rehabilitation (U01-U54)',
       'V Bones & Joints Of Skull & Spine (V01-V69)',
       'W Other Bones & Joints (W01-W99,O06-O10, O17-O19, O21-O27, O29, O32, O35, O37-O40)',
       'X Miscellaneous Operations (X01-X97)'], dtype=obje

In [52]:
all_procs_mask = procedures_df['procedure_group'].str.contains('all procedures', case=False, na=False)


In [53]:
procedures_df.loc[all_procs_mask, ['procedure_group', 'male_fce', 'female_fce']]

,procedure_group,male_fce,female_fce
1,All Procedures,9609513,11687625


In [54]:
proc_analysis_df = procedures_df[~(subheading_mask | all_procs_mask)].copy()

In [55]:
proc_analysis_df.loc[
    proc_analysis_df['procedure_group'].str.match(r'^[A-Z]\s', na=False),
    'procedure_group'
]

Series([], Name: procedure_group, dtype: object)

In [56]:
proc_analysis_df['procedure_group'].head(20)

3                          A1 Tissue of brain (A01-A11)
4     A2 Ventricle of brain and subarachnoid space (...
5                           A3 Cranial nerves (A24-A36)
6                        A4 Meninges of brain (A38-A43)
7     A5 Spinal cord and other contents of spinal ca...
8                        A6 Peripheral nerves (A59-A73)
9            A7 Other parts of nervous system (A75-A84)
11             B1 Pituitary and pineal glands (B01-B06)
12          B2 Thyroid and parathyroid glands (B08-B17)
13                  B3 Other endocrine glands (B18-B25)
14                                  B4 Breast (B27-B41)
16                                   C1 Orbit (C01-C08)
17                      C2 Eyebrow and eyelid (C09-C23)
18                      C3 Lacrimal apparatus (C24-C29)
19                          C4 Muscles of eye (C31-C37)
20                  C5 Conjunctiva and cornea (C39-C51)
21                         C6 Sclera and iris (C52-C65)
22        C7 Anterior chamber of eye and lens (C

In [57]:
sum_male = proc_analysis_df['male_fce'].sum()
sum_female = proc_analysis_df['female_fce'].sum()

sum_male, sum_female

(np.int64(5830142), np.int64(6863739))

In [58]:
print(proc_analysis_df.head(20))
print(proc_analysis_df.tail(20))

                                      procedure_group  male_fce  female_fce
3                        A1 Tissue of brain (A01-A11)      5491        4382
4   A2 Ventricle of brain and subarachnoid space (...      3706        3695
5                         A3 Cranial nerves (A24-A36)      1462        1851
6                      A4 Meninges of brain (A38-A43)      3541        2663
7   A5 Spinal cord and other contents of spinal ca...     61877       79476
8                      A6 Peripheral nerves (A59-A73)     43894       72041
9          A7 Other parts of nervous system (A75-A84)     10882       10523
11           B1 Pituitary and pineal glands (B01-B06)       771         699
12        B2 Thyroid and parathyroid glands (B08-B17)      4632       15025
13                B3 Other endocrine glands (B18-B25)       863         926
14                                B4 Breast (B27-B41)      1633       88348
16                                 C1 Orbit (C01-C08)      1665        1323
17          

In [59]:
proc_analysis_df.to_csv('cleaned_procedures_analysis.csv', index=False)

In [60]:
# checking outputted file
pd.read_csv('../data/clean/cleaned_procedures_analysis.csv').head(20)

,procedure_group,male_fce,female_fce
0,A1 Tissue of brain (A01-A11),5491,4382
1,A2 Ventricle of brain and subarachnoid space (...,3706,3695
2,A3 Cranial nerves (A24-A36),1462,1851
3,A4 Meninges of brain (A38-A43),3541,2663
4,A5 Spinal cord and other contents of spinal ca...,61877,79476
5,A6 Peripheral nerves (A59-A73),43894,72041
6,A7 Other parts of nervous system (A75-A84),10882,10523
7,B1 Pituitary and pineal glands (B01-B06),771,699
8,B2 Thyroid and parathyroid glands (B08-B17),4632,15025
9,B3 Other endocrine glands (B18-B25),863,926


### Hospital admitted patient care 2023-24 - Patient demographics

In [61]:
demog_file_path = '../data/raw/hosp-epis-stat-admi-demog-2023-24-tab.xlsx'

In [62]:
# Using the 'Ethnic Category' sheet from the demographics workbook
# Checking the raw structure to see where the real header and data rows are

demog_raw_df = pd.read_excel(
    demog_file_path,
    sheet_name='Ethnic Category',
    engine='openpyxl',
    header=None
)

demog_raw_df.head(15)

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Ethnic Category, 2023-24",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,This table groups episodes by the ethnicity of...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,"Copyright © 2024, NHS England.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Activity in English NHS Hospitals and English ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,For more information on data quality please se...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Zero bed day cases §,NaN,NaN


In [63]:
# Header row with real column names is at row 10 and the actual data starts at row 12

header_row_index = 10
header_row = demog_raw_df.iloc[header_row_index]
data_start_index = 12
demog_df = demog_raw_df.iloc[data_start_index:].copy()
demog_df.columns = header_row
demog_df.head()

10,Ethnic Category,NaN,NaN,NaN,NaN,NaN,NaN,Finished consultant episodes,Finished Admission Episodes,Male \n(FCE),...,Age 70-74 \n(FCE),Age 75-79 \n(FCE),Age 80-84 \n(FCE),Age 85-89 \n(FCE),Age 90+ \n(FCE),Day case \n(FCE),FCE bed days,Emergency \n(FAE),Elective\n(FAE),Other\n(FAE)
12,Total,NaN,NaN,NaN,NaN,NaN,NaN,21450875,17560128,9609513,...,1854892,2183072,1692024,1272957,790982,7959024,48324062,2419256,196436,630653
13,A,British (White),NaN,NaN,NaN,NaN,NaN,14647984,11765955,6605131,...,1391173,1708295,1329067,1017957,642864,5350935,34069712,1633308,130711,364297
14,B,Irish (White),NaN,NaN,NaN,NaN,NaN,143887,110968,65525,...,15958,20192,18750,13857,7144,52821,385052,12052,1258,3569
15,C,Any other White background,NaN,NaN,NaN,NaN,NaN,963874,816817,391041,...,47751,50451,39436,31163,21683,299527,2192309,122896,8434,54952
16,D,White and Black Caribbean (Mixed),NaN,NaN,NaN,NaN,NaN,70872,62743,25736,...,1415,1209,1206,868,495,19107,161600,10234,788,5781


In [64]:
# Checking full list of column names to decide what to keep

demog_df.columns.tolist()

['Ethnic Category',
 nan,
 np.float64(nan),
 np.float64(nan),
 np.float64(nan),
 np.float64(nan),
 np.float64(nan),
 'Finished consultant episodes',
 'Finished Admission Episodes',
 'Male \n(FCE)',
 'Female \n(FCE)',
 'Gender Unknown \n(FCE)',
 'Emergency \n(FAE)',
 'Waiting list \n(FAE)',
 'Planned (FAE)',
 'Other (FAE)',
 'Mean time waited \n(Days)',
 'Median time waited \n(Days)',
 'Mean length of stay \n(Days)',
 'Median length of stay \n(Days)',
 'Mean age \n(Years)',
 'Age 0 \n(FCE)',
 'Age 1-4 \n(FCE)',
 'Age 5-9 \n(FCE)',
 'Age 10-14 \n(FCE)',
 'Age 15 \n(FCE)',
 'Age 16 \n(FCE)',
 'Age 17 \n(FCE)',
 'Age 18 \n(FCE)',
 'Age 19 \n(FCE)',
 'Age 20-24 \n(FCE)',
 'Age 25-29 \n(FCE)',
 'Age 30-34 \n(FCE)',
 'Age 35-39 \n(FCE)',
 'Age 40-44 \n(FCE)',
 'Age 45-49 \n(FCE)',
 'Age 50-54 \n(FCE)',
 'Age 55-59 \n(FCE)',
 'Age 60-64 \n(FCE)',
 'Age 65-69 \n(FCE)',
 'Age 70-74 \n(FCE)',
 'Age 75-79 \n(FCE)',
 'Age 80-84 \n(FCE)',
 'Age 85-89 \n(FCE)',
 'Age 90+ \n(FCE)',
 'Day case \n(FCE)'

In [65]:
cols = demog_df.columns.tolist()

ethnic_code_col = cols[0]
ethnic_desc_col = cols[1]

# Required for men vs women comparison
male_fce_col = 'Male \n(FCE)'
female_fce_col = 'Female \n(FCE)'

# May be useful for context
total_fce_col = 'Finished consultant episodes'
total_fae_col = 'Finished Admission Episodes'

# Creating a smaller dataframe using only these columns
demog_sex_df = demog_df[[ethnic_code_col,
                         ethnic_desc_col,
                         total_fce_col,
                         total_fae_col,
                         male_fce_col,
                         female_fce_col]].copy()

# Renaming columns
demog_sex_df = demog_sex_df.rename(columns={
    ethnic_code_col: 'Ethnic_Code',
    ethnic_desc_col: 'Ethnic_Description',
    total_fce_col: 'Total_FCE',
    total_fae_col: 'Total_FAE',
    male_fce_col: 'Male_FCE',
    female_fce_col: 'Female_FCE'
})

demog_sex_df.head()

10,Ethnic_Code,Ethnic_Description,NaN,NaN,NaN,NaN,NaN,Total_FCE,Total_FAE,Male_FCE,Female_FCE
12,Total,NaN,NaN,NaN,NaN,NaN,NaN,21450875,17560128,9609513,11687625
13,A,British (White),NaN,NaN,NaN,NaN,NaN,14647984,11765955,6605131,7955166
14,B,Irish (White),NaN,NaN,NaN,NaN,NaN,143887,110968,65525,77279
15,C,Any other White background,NaN,NaN,NaN,NaN,NaN,963874,816817,391041,563634
16,D,White and Black Caribbean (Mixed),NaN,NaN,NaN,NaN,NaN,70872,62743,25736,44619


In [66]:
# Checking for missing values
demog_sex_df.isnull().sum()

10
Ethnic_Code            2
Ethnic_Description     4
NaN                   22
NaN                   22
NaN                   22
NaN                   22
NaN                   22
Total_FCE              3
Total_FAE              3
Male_FCE               3
Female_FCE             3
dtype: int64

In [67]:
# Convert count columns to numeric (they may be different format in excel)

numeric_cols = ['Total_FCE', 'Total_FAE', 'Male_FCE', 'Female_FCE']

for col in numeric_cols:
  demog_sex_df[col] = pd.to_numeric(demog_sex_df[col], errors='coerce')

demog_sex_df.dtypes

10
Ethnic_Code            object
Ethnic_Description     object
NaN                   float64
NaN                   float64
NaN                   float64
NaN                   float64
NaN                   float64
Total_FCE             float64
Total_FAE             float64
Male_FCE              float64
Female_FCE            float64
dtype: object

In [68]:
# Drop rows where all numeric columns are NaN and check the result

demog_sex_df = demog_sex_df.dropna(subset=numeric_cols, how='all')

demog_sex_df.head()
demog_sex_df.tail()

10,Ethnic_Code,Ethnic_Description,NaN,NaN,NaN,NaN,NaN,Total_FCE,Total_FAE,Male_FCE,Female_FCE
26,P,Any other Black background,NaN,NaN,NaN,NaN,NaN,134575.0,110524.0,57617.0,75006.0
27,R,Chinese (other ethnic group),NaN,NaN,NaN,NaN,NaN,56438.0,48384.0,22369.0,33671.0
28,S,Any other ethnic group,NaN,NaN,NaN,NaN,NaN,466736.0,392160.0,209238.0,253247.0
29,Z,Not stated,NaN,NaN,NaN,NaN,NaN,1880988.0,1604525.0,903197.0,954920.0
30,99,Not known,NaN,NaN,NaN,NaN,NaN,991325.0,869500.0,478887.0,498693.0


In [69]:
numeric_cols = ['Total_FCE', 'Total_FAE', 'Male_FCE', 'Female_FCE']

In [70]:
# Restarted the process by reloading the raw demographics sheet from the excel file
# Lost data when trying to clean the table further to remove columns with Nan

demog_raw_df = pd.read_excel(
    demog_file_path,
    sheet_name='Ethnic Category',
    engine='openpyxl',
    header=None
)

demog_raw_df.head(15)

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Ethnic Category, 2023-24",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,This table groups episodes by the ethnicity of...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,"Copyright © 2024, NHS England.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Activity in English NHS Hospitals and English ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,For more information on data quality please se...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Zero bed day cases §,NaN,NaN


In [71]:
header_row_index = 10
data_start_index = 12

header_row = demog_raw_df.iloc[header_row_index]

demog_df = demog_raw_df.iloc[data_start_index:].copy()

demog_df.columns = header_row

demog_df.head()

10,Ethnic Category,NaN,NaN,NaN,NaN,NaN,NaN,Finished consultant episodes,Finished Admission Episodes,Male \n(FCE),...,Age 70-74 \n(FCE),Age 75-79 \n(FCE),Age 80-84 \n(FCE),Age 85-89 \n(FCE),Age 90+ \n(FCE),Day case \n(FCE),FCE bed days,Emergency \n(FAE),Elective\n(FAE),Other\n(FAE)
12,Total,NaN,NaN,NaN,NaN,NaN,NaN,21450875,17560128,9609513,...,1854892,2183072,1692024,1272957,790982,7959024,48324062,2419256,196436,630653
13,A,British (White),NaN,NaN,NaN,NaN,NaN,14647984,11765955,6605131,...,1391173,1708295,1329067,1017957,642864,5350935,34069712,1633308,130711,364297
14,B,Irish (White),NaN,NaN,NaN,NaN,NaN,143887,110968,65525,...,15958,20192,18750,13857,7144,52821,385052,12052,1258,3569
15,C,Any other White background,NaN,NaN,NaN,NaN,NaN,963874,816817,391041,...,47751,50451,39436,31163,21683,299527,2192309,122896,8434,54952
16,D,White and Black Caribbean (Mixed),NaN,NaN,NaN,NaN,NaN,70872,62743,25736,...,1415,1209,1206,868,495,19107,161600,10234,788,5781


In [72]:
# Drop columns that are empty

demog_df = demog_df.dropna(axis=1, how='all')

demog_df.head()
demog_df.columns.tolist()

['Ethnic Category',
 nan,
 'Finished consultant episodes',
 'Finished Admission Episodes',
 'Male \n(FCE)',
 'Female \n(FCE)',
 'Gender Unknown \n(FCE)',
 'Emergency \n(FAE)',
 'Waiting list \n(FAE)',
 'Planned (FAE)',
 'Other (FAE)',
 'Mean time waited \n(Days)',
 'Median time waited \n(Days)',
 'Mean length of stay \n(Days)',
 'Median length of stay \n(Days)',
 'Mean age \n(Years)',
 'Age 0 \n(FCE)',
 'Age 1-4 \n(FCE)',
 'Age 5-9 \n(FCE)',
 'Age 10-14 \n(FCE)',
 'Age 15 \n(FCE)',
 'Age 16 \n(FCE)',
 'Age 17 \n(FCE)',
 'Age 18 \n(FCE)',
 'Age 19 \n(FCE)',
 'Age 20-24 \n(FCE)',
 'Age 25-29 \n(FCE)',
 'Age 30-34 \n(FCE)',
 'Age 35-39 \n(FCE)',
 'Age 40-44 \n(FCE)',
 'Age 45-49 \n(FCE)',
 'Age 50-54 \n(FCE)',
 'Age 55-59 \n(FCE)',
 'Age 60-64 \n(FCE)',
 'Age 65-69 \n(FCE)',
 'Age 70-74 \n(FCE)',
 'Age 75-79 \n(FCE)',
 'Age 80-84 \n(FCE)',
 'Age 85-89 \n(FCE)',
 'Age 90+ \n(FCE)',
 'Day case \n(FCE)',
 'FCE bed days',
 'Emergency \n(FAE)',
 'Elective\n(FAE)',
 'Other\n(FAE)']

In [73]:
# Built a new smaller data frame again and renamed columns

ethnic_code_col = cols[0]

ethnic_desc_col = cols[1]

total_fce_col  = 'Finished consultant episodes'
total_fae_col  = 'Finished Admission Episodes'
male_fce_col   = 'Male \n(FCE)'
female_fce_col = 'Female \n(FCE)'

demog_sex_df = demog_df[
    [ethnic_code_col,
     ethnic_desc_col,
     total_fce_col,
     total_fae_col,
     male_fce_col,
     female_fce_col]
].copy()

demog_sex_df = demog_sex_df.rename(columns={
    ethnic_code_col: 'Ethnic_Code',
    ethnic_desc_col: 'Ethnic_Description',
    total_fce_col:   'Total_FCE',
    total_fae_col:   'Total_FAE',
    male_fce_col:    'Male_FCE',
    female_fce_col:  'Female_FCE'
})

demog_sex_df.head()

10,Ethnic_Code,Ethnic_Description,Total_FCE,Total_FAE,Male_FCE,Female_FCE
12,Total,NaN,21450875,17560128,9609513,11687625
13,A,British (White),14647984,11765955,6605131,7955166
14,B,Irish (White),143887,110968,65525,77279
15,C,Any other White background,963874,816817,391041,563634
16,D,White and Black Caribbean (Mixed),70872,62743,25736,44619


In [74]:
# Remove columns with NaN, drop the 'Total' row

numeric_cols = ['Total_FCE', 'Total_FAE', 'Male_FCE', 'Female_FCE']

demog_sex_df = demog_sex_df.dropna(subset=numeric_cols, how='all')

demog_sex_df = demog_sex_df[demog_sex_df['Ethnic_Code'] != 'Total']

for col in numeric_cols:
    demog_sex_df[col] = pd.to_numeric(demog_sex_df[col], errors='coerce')

demog_sex_df.head(), demog_sex_df.tail()

(10 Ethnic_Code                 Ethnic_Description  Total_FCE  Total_FAE  \
 13           A                    British (White)   14647984   11765955   
 14           B                      Irish (White)     143887     110968   
 15           C         Any other White background     963874     816817   
 16           D  White and Black Caribbean (Mixed)      70872      62743   
 17           E    White and Black African (Mixed)      38271      33925   
 
 10  Male_FCE  Female_FCE  
 13   6605131     7955166  
 14     65525       77279  
 15    391041      563634  
 16     25736       44619  
 17     14997       22887  ,
 10 Ethnic_Code            Ethnic_Description  Total_FCE  Total_FAE  Male_FCE  \
 26           P    Any other Black background     134575     110524     57617   
 27           R  Chinese (other ethnic group)      56438      48384     22369   
 28           S        Any other ethnic group     466736     392160    209238   
 29           Z                    Not stated    

In [75]:
demog_sex_df.head()

10,Ethnic_Code,Ethnic_Description,Total_FCE,Total_FAE,Male_FCE,Female_FCE
13,A,British (White),14647984,11765955,6605131,7955166
14,B,Irish (White),143887,110968,65525,77279
15,C,Any other White background,963874,816817,391041,563634
16,D,White and Black Caribbean (Mixed),70872,62743,25736,44619
17,E,White and Black African (Mixed),38271,33925,14997,22887


In [76]:
demog_sex_df.to_csv('demographics_ethnic_sex_clean.csv', index=False)

## Patient Level Activity and Costing 2021-2022

### Exploring the Data

In [77]:
file_path = "../data/raw/Integrated-Patient-Level-Activity-and-Costing-2021-22.csv"
costing_21_22_df = pd.read_csv(file_path)

costing_21_22_df.head()

,YEAR,DATA_SET,ORG_LEVEL,ORG_CODE,ORG_NAME,BREAKDOWN,BREAKDOWN_GROUP_1,BREAKDOWN_GROUP_2,ACTIVITY_COUNT,TOTAL_COST
0,202122,APC,ALL,ALL,ALL SUBMITTERS,Diagnosis,Diseases of the skin and subcutaneous tissue,NaN,363861.000000,6.102477e+08
1,202122,APC,ALL,ALL,ALL SUBMITTERS,Diagnosis,"Endocrine, nutritional and metabolic diseases",NaN,427412.000000,7.211952e+08
2,202122,IAPT,ALL,ALL,ALL SUBMITTERS,Cluster Assessment Status and Cluster,Patient assessed and assigned an Adult Mental ...,06 - Non-Psychotic Disorder of Over-Valued Ideas,9938.000000,1.731191e+06
3,202122,IAPT,ALL,ALL,ALL SUBMITTERS,Cluster Assessment Status and Cluster,Patient assessed and assigned an Adult Mental ...,07 - Enduring Non-Psychotic Disorders (High Di...,12747.000000,2.147146e+06
4,202122,IAPT,ALL,ALL,ALL SUBMITTERS,Cluster Assessment Status and Cluster,Patient assessed and assigned an Adult Mental ...,08 - Non-Psychotic Chaotic and Challenging Dis...,2871.000000,4.320896e+05


**Exploring data shape and type:**

In [78]:
costing_21_22_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48429 entries, 0 to 48428
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   YEAR               48429 non-null  int64  
 1   DATA_SET           48429 non-null  object 
 2   ORG_LEVEL          48429 non-null  object 
 3   ORG_CODE           48429 non-null  object 
 4   ORG_NAME           48429 non-null  object 
 5   BREAKDOWN          48429 non-null  object 
 6   BREAKDOWN_GROUP_1  46996 non-null  object 
 7   BREAKDOWN_GROUP_2  36276 non-null  object 
 8   ACTIVITY_COUNT     48429 non-null  object 
 9   TOTAL_COST         48429 non-null  float64
dtypes: float64(1), int64(1), object(8)
memory usage: 3.7+ MB


In [79]:
costing_21_22_df.columns

Index(['YEAR', 'DATA_SET', 'ORG_LEVEL', 'ORG_CODE', 'ORG_NAME', 'BREAKDOWN',
       'BREAKDOWN_GROUP_1', 'BREAKDOWN_GROUP_2', 'ACTIVITY_COUNT',
       'TOTAL_COST'],
      dtype='object')

In [80]:
costing_21_22_df.shape [0]

48429

In [7]:
#Here I'm selecting which columns to include and dropping the remainder irrelevant ones
costing_21_22_df["BREAKDOWN_GROUP_1"] = costing_21_22_df["BREAKDOWN_GROUP_1"].astype(str).str.strip()
costing_21_22_df["BREAKDOWN_GROUP_2"] = costing_21_22_df["BREAKDOWN_GROUP_2"].astype(str).str.strip()

costing_21_22_df["ACTIVITY_COUNT"] = pd.to_numeric(costing_21_22_df["ACTIVITY_COUNT"], errors="coerce")
costing_21_22_df["TOTAL_COST"] = pd.to_numeric(costing_21_22_df["TOTAL_COST"], errors="coerce")


costing_21_22_df = costing_21_22_df.dropna(subset=["ACTIVITY_COUNT", "TOTAL_COST"], how="all")

costing_21_22_df.head()

,YEAR,DATA_SET,ORG_LEVEL,ORG_CODE,ORG_NAME,BREAKDOWN,BREAKDOWN_GROUP_1,BREAKDOWN_GROUP_2,ACTIVITY_COUNT,TOTAL_COST
0,202122,APC,ALL,ALL,ALL SUBMITTERS,Diagnosis,Diseases of the skin and subcutaneous tissue,nan,363861.0,6.102477e+08
1,202122,APC,ALL,ALL,ALL SUBMITTERS,Diagnosis,"Endocrine, nutritional and metabolic diseases",nan,427412.0,7.211952e+08
2,202122,IAPT,ALL,ALL,ALL SUBMITTERS,Cluster Assessment Status and Cluster,Patient assessed and assigned an Adult Mental ...,06 - Non-Psychotic Disorder of Over-Valued Ideas,9938.0,1.731191e+06
3,202122,IAPT,ALL,ALL,ALL SUBMITTERS,Cluster Assessment Status and Cluster,Patient assessed and assigned an Adult Mental ...,07 - Enduring Non-Psychotic Disorders (High Di...,12747.0,2.147146e+06
4,202122,IAPT,ALL,ALL,ALL SUBMITTERS,Cluster Assessment Status and Cluster,Patient assessed and assigned an Adult Mental ...,08 - Non-Psychotic Chaotic and Challenging Dis...,2871.0,4.320896e+05


### Cleaning the data

**Step 1**: Dropping columns down to relevance (48,429 rows)
- no extra spaces
- no symbols
- Standardised format of values ie in Accounting format with £ symbols
- ensuring column characters are in the same format i.e. all uppercase

In [87]:
# filter to male and female only
gender_df = costing_21_22_df[costing_21_22_df["BREAKDOWN_GROUP_1"].isin(["Male", "Female"])].copy()

gender_df["ACTIVITY_COUNT"] = pd.to_numeric(gender_df["ACTIVITY_COUNT"], errors="coerce")
gender_df["TOTAL_COST"] = pd.to_numeric(gender_df["TOTAL_COST"], errors="coerce")

# keeps only the required columns
gender_df = gender_df[[
    "BREAKDOWN",
    "BREAKDOWN_GROUP_1",
    "BREAKDOWN_GROUP_2",
    "ACTIVITY_COUNT",
    "TOTAL_COST"
]].copy()

gender_summary = (
    gender_df
    .groupby("BREAKDOWN_GROUP_1", as_index=False)[["ACTIVITY_COUNT", "TOTAL_COST"]]
    .sum()
)
#reformat ACTIVITY_COUNT and TOTAL_COST
gender_df["ACTIVITY_COUNT"] = (
    gender_df["ACTIVITY_COUNT"]
    .fillna(0)
    .round(0)
    .astype(int)
    .map("{:,}".format)
)

gender_df["TOTAL_COST"] = (
    gender_df["TOTAL_COST"]
    .fillna(0)
    .round(0)
    .astype(int)
    .map("£{:,.0f}".format)
)


gender_df.head(20)

,BREAKDOWN,BREAKDOWN_GROUP_1,BREAKDOWN_GROUP_2,ACTIVITY_COUNT,TOTAL_COST
6,Sex and Age Group,Female,80-84 years,"642,609","£1,609,173,362"
7,Sex and Age Group,Female,85-89 years,"538,576","£1,457,331,396"
8,Sex and Age Group,Female,90-94 years,"307,462","£887,009,480"
37,Sex and Age Group,Female,0 years,"208,352","£35,405,523"
38,Sex and Age Group,Female,01-04 years,"536,027","£91,711,675"
39,Sex and Age Group,Male,30-34 years,"218,303","£383,703,947"
40,Sex and Age Group,Male,35-39 years,"239,127","£436,654,049"
41,Sex and Age Group,Male,40-44 years,"264,984","£485,662,707"
42,Sex and Age Group,Male,45-49 years,"307,961","£567,549,934"
53,Sex and Age Group,Female,50-54 years,"2,713,691","£470,693,787"


### Preview Changes Made

**Step 2:**
- Trimming down to just 3 columns telling us about overall usage and costings for Men vs Women

In [88]:
#Format numbers nicely

summary_reformatted = gender_summary.copy()

summary_reformatted["ACTIVITY_COUNT"] = (
    summary_reformatted["ACTIVITY_COUNT"]
    .round(0)
    .astype(int)
    .map("{:,}".format)
)

summary_reformatted["TOTAL_COST"] = (
    summary_reformatted["TOTAL_COST"]
    .round(0)
    .astype(int)
    .map("£{:,.0f}".format)
)

summary_reformatted

,BREAKDOWN_GROUP_1,ACTIVITY_COUNT,TOTAL_COST
0,Female,"233,928,897","£112,370,895,478"
1,Male,"179,435,584","£91,843,605,253"


### Interpretation

**Women:**
- Cost more on the NHS across the span of their life overall
- Increases in cost appear more in their 'child-bearing' years suggesting our hypothesis is correct in that greater costs are likely linked to maternity care

**Men:**
- Cost less on the NHS across the span of their life overall
- Increases in cost appear more in their later years in life as they age

### Saving cleaned dataset to csv

In [138]:
costing_21_22_df.to_csv("patient_level_activity_cleaned.csv", index=False)

**Summary**

- Covers two time periods yrs '21 and '22
-  5 usable columns
-  Over 1.2 million rows of raw data, with 48,429 rows in the cleaned dataset (4% of whole)

**Columns available:**
- Breakdown (Age, Sex)
- Activity Count
- Total Cost
- Breakdown Group 1 (Sex)
- Breakdown Group 2 (Age )

## Acute Patient Level Activity and Costing 2019-2020

In [89]:
file_path = "../data/raw/Acute-Patient-Level-Activity-and-Costing-2019-20.csv"
costing_19_20_df = pd.read_csv(file_path)

costing_19_20_df.head()

,YEAR,DATA_SET,ORG_LEVEL,ORG_CODE,ORG_NAME,BREAKDOWN,BREAKDOWN_GROUP_1,BREAKDOWN_GROUP_2,SUMMARY_TYPE,VALUE
0,201920,AE,ALL,ALL,ALL SUBMITTERS,Total Activities,NaN,NaN,Activity Count,18082625
1,201920,AE,ALL,ALL,ALL SUBMITTERS,Total Cost,NaN,NaN,Total Cost,3494794626
2,201920,AE,ALL,ALL,ALL SUBMITTERS,Gender and Age Group,Female,0 years,Activity Count,212079
3,201920,AE,ALL,ALL,ALL SUBMITTERS,Gender and Age Group,Female,01-04 years,Activity Count,505885
4,201920,AE,ALL,ALL,ALL SUBMITTERS,Gender and Age Group,Female,05-09 years,Activity Count,347081


In [91]:
# Cleaning column headers
costing_19_20_df["BREAKDOWN_GROUP_1"] = costing_19_20_df["BREAKDOWN_GROUP_1"].astype(str).str.strip()
costing_19_20_df["BREAKDOWN_GROUP_2"] = costing_19_20_df["BREAKDOWN_GROUP_2"].astype(str).str.strip()

# ensuring data in value row is numeric
costing_19_20_df["VALUE"] = pd.to_numeric(costing_19_20_df["VALUE"], errors="coerce")

# dropping missing data in 'value' column to remove unnecessary rows
costing_19_20_df = costing_19_20_df.dropna(subset=["VALUE"], how="all")

# convert numbers to int
costing_19_20_df["VALUE"] = costing_19_20_df["VALUE"].astype(int)

costing_19_20_df.head()

,YEAR,DATA_SET,ORG_LEVEL,ORG_CODE,ORG_NAME,BREAKDOWN,BREAKDOWN_GROUP_1,BREAKDOWN_GROUP_2,SUMMARY_TYPE,VALUE
0,201920,AE,ALL,ALL,ALL SUBMITTERS,Total Activities,nan,nan,Activity Count,18082625
1,201920,AE,ALL,ALL,ALL SUBMITTERS,Total Cost,nan,nan,Total Cost,3494794626
2,201920,AE,ALL,ALL,ALL SUBMITTERS,Gender and Age Group,Female,0 years,Activity Count,212079
3,201920,AE,ALL,ALL,ALL SUBMITTERS,Gender and Age Group,Female,01-04 years,Activity Count,505885
4,201920,AE,ALL,ALL,ALL SUBMITTERS,Gender and Age Group,Female,05-09 years,Activity Count,347081


In [92]:
# focusing dataframe on male & female records only
costing_19_20_df = costing_19_20_df[costing_19_20_df["BREAKDOWN_GROUP_1"].isin(["Male", "Female"])].copy()
costing_19_20_df.head()
# costing_19_20_df.tail()

,YEAR,DATA_SET,ORG_LEVEL,ORG_CODE,ORG_NAME,BREAKDOWN,BREAKDOWN_GROUP_1,BREAKDOWN_GROUP_2,SUMMARY_TYPE,VALUE
2,201920,AE,ALL,ALL,ALL SUBMITTERS,Gender and Age Group,Female,0 years,Activity Count,212079
3,201920,AE,ALL,ALL,ALL SUBMITTERS,Gender and Age Group,Female,01-04 years,Activity Count,505885
4,201920,AE,ALL,ALL,ALL SUBMITTERS,Gender and Age Group,Female,05-09 years,Activity Count,347081
5,201920,AE,ALL,ALL,ALL SUBMITTERS,Gender and Age Group,Female,10-14 years,Activity Count,371563
6,201920,AE,ALL,ALL,ALL SUBMITTERS,Gender and Age Group,Female,15-19 years,Activity Count,483626


In [93]:
# removing unnecessary columns
costing_19_20_df = costing_19_20_df[[
    "BREAKDOWN",
    "BREAKDOWN_GROUP_1",
    "BREAKDOWN_GROUP_2",
    "VALUE"
]].copy()
costing_19_20_df.head()

,BREAKDOWN,BREAKDOWN_GROUP_1,BREAKDOWN_GROUP_2,VALUE
2,Gender and Age Group,Female,0 years,212079
3,Gender and Age Group,Female,01-04 years,505885
4,Gender and Age Group,Female,05-09 years,347081
5,Gender and Age Group,Female,10-14 years,371563
6,Gender and Age Group,Female,15-19 years,483626


In [94]:
# rename 'value' column to match 2021-22 data, to make combining easier
costing_19_20_df = costing_19_20_df.rename(columns={"VALUE": "TOTAL_COST"})
costing_19_20_df.head()

,BREAKDOWN,BREAKDOWN_GROUP_1,BREAKDOWN_GROUP_2,TOTAL_COST
2,Gender and Age Group,Female,0 years,212079
3,Gender and Age Group,Female,01-04 years,505885
4,Gender and Age Group,Female,05-09 years,347081
5,Gender and Age Group,Female,10-14 years,371563
6,Gender and Age Group,Female,15-19 years,483626


In [95]:
costing_19_20_df.shape

(37588, 4)

After cleaning we have:

* Rows: 37588
* Columns: 4

The columns do not match the 2021-22 data, as this dataset is missing the activity count column. However the two sets of years can be compared in the analysis phase.

In [28]:
# saving cleaned data to csv
costing_19_20_df.to_csv("patient_level_activity_2019_20_cleaned.csv", index=False)

## UK Health Accounts - 2024

UK healthcare expenditure data by financing scheme, function and provider

### Current healthcare expenditure by financing scheme, in nominal terms, £ millions, 1997 to 2024

In [96]:
current_expenditure_df = pd.read_excel('../data/raw/ukhareferencetablespublished20250430.xlsx', engine="openpyxl", sheet_name='1a')
current_expenditure_df.head()

,"Table 1a: Current healthcare expenditure [note 1] by financing scheme [note 2], in nominal terms [note 3], £ millions, 1997 to 2024",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29
0,There is one table on this worksheet.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,United Kingdom,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ICHA codes [note 4],Financing Scheme,1997.0,1998.0,1999.0,2000.0,2001.0,2002.0,2003.0,2004.0,...,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024 [note 10]
3,HF.TOT,Total current healthcare expenditure,65342.0,69626.0,74045.0,78655.0,85190.0,93641.0,101323.0,110072.0,...,190504.0,196592.0,202098.0,210817.0,222935.0,254603.0,276156.0,280353.0,297994.0,317411
4,HF.1.1,Government-financed expenditure [note 5],48381.0,51895.0,56075.0,59304.0,64301.0,71529.0,78902.0,87754.0,...,150363.0,156171.0,159453.0,165931.0,176196.0,212607.0,229762.0,228541.0,242035.0,258060


In [97]:
# Applying column names using values in row index 2
expenditure_columns = current_expenditure_df.loc[2, :].values.flatten().tolist()
# Converting column names in the list to strings
expenditure_columns = [str(col) for col in expenditure_columns]
current_expenditure_df.columns = expenditure_columns
current_expenditure_df.head()

,ICHA codes [note 4],Financing Scheme,1997.0,1998.0,1999.0,2000.0,2001.0,2002.0,2003.0,2004.0,...,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024 [note 10]
0,There is one table on this worksheet.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,United Kingdom,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ICHA codes [note 4],Financing Scheme,1997.0,1998.0,1999.0,2000.0,2001.0,2002.0,2003.0,2004.0,...,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024 [note 10]
3,HF.TOT,Total current healthcare expenditure,65342.0,69626.0,74045.0,78655.0,85190.0,93641.0,101323.0,110072.0,...,190504.0,196592.0,202098.0,210817.0,222935.0,254603.0,276156.0,280353.0,297994.0,317411
4,HF.1.1,Government-financed expenditure [note 5],48381.0,51895.0,56075.0,59304.0,64301.0,71529.0,78902.0,87754.0,...,150363.0,156171.0,159453.0,165931.0,176196.0,212607.0,229762.0,228541.0,242035.0,258060


In [98]:
# clean column names
current_expenditure_df.columns = (current_expenditure_df.columns
                                  .str.strip()
                                  .str.lower()
                                  .str.replace('\n', '')
                                  .str.replace(' ', '_')
                                  .str.replace('.0', '', regex=False))

# rename 2024 column
current_expenditure_df.rename(columns={'2024__[note_10]' : '2024'}, inplace = True)

# Clean the 'financing_scheme' column
current_expenditure_df['financing_scheme'] = current_expenditure_df['financing_scheme'].astype(str).str.split('[').str[0]

# drop unneeded columns
current_expenditure_clean = current_expenditure_df.drop(index=[0, 1, 2]).copy()
current_expenditure_clean.dropna(how='any', inplace=True)

# Drop the 'icha_codes_[note_4]' column
current_expenditure_clean = current_expenditure_clean.drop("icha_codes_[note_4]", axis='columns')

current_expenditure_clean.head()

,financing_scheme,1997,1998,1999,2000,2001,2002,2003,2004,2005,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
3,Total current healthcare expenditure,65342.0,69626.0,74045.0,78655.0,85190.0,93641.0,101323.0,110072.0,117230.0,...,190504.0,196592.0,202098.0,210817.0,222935.0,254603.0,276156.0,280353.0,297994.0,317411
4,Government-financed expenditure,48381.0,51895.0,56075.0,59304.0,64301.0,71529.0,78902.0,87754.0,94758.0,...,150363.0,156171.0,159453.0,165931.0,176196.0,212607.0,229762.0,228541.0,242035.0,258060
5,Voluntary health insurance schemes,2462.0,2580.0,2761.0,3346.0,3775.0,3961.0,4045.0,4189.0,4640.0,...,6143.0,5444.0,5956.0,6128.0,6329.0,6002.0,6127.0,7056.0,7723.0,8344
6,Non-profit institutions serving households fin...,721.0,802.0,887.0,997.0,1077.0,1210.0,1379.0,1389.0,1511.0,...,3135.0,3532.0,4077.0,4022.0,4018.0,3921.0,3866.0,3998.0,4166.0,4308
7,Enterprise financing schemes,713.0,749.0,646.0,680.0,662.0,659.0,630.0,562.0,577.0,...,638.0,620.0,623.0,630.0,640.0,477.0,436.0,439.0,476.0,498


In [37]:
current_expenditure_clean.to_csv("UK_Accounts_current_expenditure_clean.csv", index=False)

### Current healthcare expenditure by financing scheme, in real terms, £ millions, 1997 to 2024

In [99]:
current_real_df = pd.read_excel('../data/raw/ukhareferencetablespublished20250430.xlsx', engine="openpyxl", sheet_name='2a')
current_real_df.head()

,"Table 2a: Current healthcare expenditure [note 1] by financing scheme [note 2], in real terms [note 3], £ millions, 1997 to 2024",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29
0,There is one table on this worksheet.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,United Kingdom,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ICHA codes [note 4],Financing Scheme,1997.0,1998.0,1999.0,2000.0,2001.0,2002.0,2003.0,2004.0,...,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024 [note 10]
3,HF.TOT,Total current healthcare expenditure,123029.0,129413.0,135726.0,142609.0,152173.0,163829.0,173115.0,183298.0,...,253719.0,256694.0,258963.0,265099.0,274570.0,298641.0,323598.0,311685.0,309914.0,317411
4,HF.1.1,Government-financed expenditure [note 5],91095.0,96458.0,102788.0,107522.0,114861.0,125143.0,134808.0,146133.0,...,200258.0,203915.0,204318.0,208654.0,217006.0,249381.0,269235.0,254083.0,251716.0,258060


In [100]:
# Applying column names using values in row index 2
real_columns = current_real_df.loc[2, :].values.flatten().tolist()
# Converting column names in the list to strings
real_columns = [str(col) for col in real_columns]
current_real_df.columns = real_columns
current_real_df.head()

# clean column names
current_real_df.columns = (current_real_df.columns
                                  .str.strip()
                                  .str.lower()
                                  .str.replace('\n', '')
                                  .str.replace(' ', '_')
                                  .str.replace('.0', '', regex=False))

# rename 2024 column
current_real_df.rename(columns={'2024__[note_10]' : '2024'}, inplace = True)

# Clean the 'financing_scheme' column
current_real_df['financing_scheme'] = current_real_df['financing_scheme'].astype(str).str.split('[').str[0]

current_real_clean = current_real_df.drop(index=[0, 1, 2]).copy()

# Drop the 'icha_codes_[note_4]' column
current_real_clean = current_real_clean.drop("icha_codes_[note_4]", axis='columns')

current_real_clean.head()

,financing_scheme,1997,1998,1999,2000,2001,2002,2003,2004,2005,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024_[note_10]
3,Total current healthcare expenditure,123029.0,129413.0,135726.0,142609.0,152173.0,163829.0,173115.0,183298.0,189716.0,...,253719.0,256694.0,258963.0,265099.0,274570.0,298641.0,323598.0,311685.0,309914.0,317411
4,Government-financed expenditure,91095.0,96458.0,102788.0,107522.0,114861.0,125143.0,134808.0,146133.0,153348.0,...,200258.0,203915.0,204318.0,208654.0,217006.0,249381.0,269235.0,254083.0,251716.0,258060
5,Voluntary health insurance schemes,4637.0,4795.0,5061.0,6066.0,6744.0,6929.0,6911.0,6976.0,7510.0,...,8182.0,7108.0,7632.0,7705.0,7795.0,7040.0,7180.0,7845.0,8032.0,8344
6,Non-profit institutions serving households fin...,1357.0,1490.0,1626.0,1808.0,1923.0,2117.0,2357.0,2313.0,2445.0,...,4176.0,4612.0,5225.0,5058.0,4949.0,4599.0,4531.0,4445.0,4332.0,4308
7,Enterprise financing schemes,1342.0,1392.0,1184.0,1233.0,1183.0,1153.0,1076.0,936.0,934.0,...,850.0,810.0,798.0,792.0,788.0,560.0,511.0,488.0,495.0,498


In [43]:
current_real_clean.to_csv('UK_Accounts_current_expenditure_real_clean.csv', index=False)

### Total current healthcare expenditure by healthcare function and healthcare provider, in nominal terms, £ millions, 2023

In [101]:
expenditure_function_df = pd.read_excel('../data/raw/ukhareferencetablespublished20250430.xlsx', engine="openpyxl", sheet_name='4a')
expenditure_function_df.head()

,"Table 4a: Total current [note 1] healthcare expenditure by healthcare function [note 2] and healthcare provider [note 3], in nominal terms [note 4], £ millions, 2023",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22
0,There is one table on this worksheet. Cells wi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,United Kingdom,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IHCA codes [note 5],[no data],HP.TOT,HP.1,HP.2,HP.3,HP.3.1,HP.3.2,HP.3.5,HP.3.x,...,HP.7,HP.7.1,HP.7.2,HP.7.3,HP.7.9,HP.8,HP.8.1,HP.8.2,HP.9,HP.0
3,[no data],Variable name,All providers,Hospitals,Residential \nlong-term care facilities,Providers of ambulatory healthcare,Offices of general medical practitioners,Dental practices,Providers of home healthcare services,Other ambulatory providers [note 6],...,Providers of healthcare system administration ...,Government health administration agencies,Social health insurance agencies,Private health insurance administration agencies,Other administration agencies,Rest of \neconomy,Households as providers of home healthcare,All other industries \nas secondary providers ...,Rest of \nthe world,Not elsewhere classified
4,HCTOT,All functions,297994,130367,37616,73131,22849,12165,15758,22359,...,4199,2349,0,1800,50,10412,4194,6218,426,2410


In [105]:
# Applying column names using values in row index 2
function_columns = expenditure_function_df.loc[3, :].values.flatten().tolist()
# Converting column names in the list to strings
expenditure_function_df.columns = function_columns
# expenditure_function_df.head()

# clean column names
expenditure_function_df.columns = (expenditure_function_df.columns
                                  .str.strip()
                                  .str.lower()
                                  .str.replace('\n', '')
                                  .str.replace(' ', '_')
                                  .str.replace('.0', '', regex=False))

# rename 'variable_name' column
expenditure_function_df.rename(columns={'variable_name' : 'healthcare_function'}, inplace = True)

# drop first 3 rows, rows with any nulls, no_data column
expenditure_function_clean = expenditure_function_df.drop(index=[0, 1, 2, 3]).copy()
expenditure_function_clean.dropna(how='any', inplace=True)
expenditure_function_clean.drop("[no_data]", axis='columns', inplace=True)

# Replace "[x]"
pd.set_option('future.no_silent_downcasting', True)
expenditure_function_clean.replace('[x]', np.nan, inplace=True)

expenditure_function_clean.sample(5)

,healthcare_function,all_providers,hospitals,residential_long-term_care_facilities,providers_of_ambulatory_healthcare,offices_of_general_medical_practitioners,dental_practices,providers_of_home_healthcare_services,other_ambulatory_providers_[note_6],providers_of_ancillary_services,...,providers_of_healthcare_system_administration_and_financing,government_health_administration_agencies,social_health_insurance_agencies,private_health_insurance_administration_agencies,other_administration_agencies,rest_of_economy,households_as_providers_of_home_healthcare,all_other_industries_as_secondary_providers_of_healthcare,rest_of_the_world,not_elsewhere_classified
26,Prescribed medicines,17548,0,0,970,970,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
28,Other medical non-durable goods,2951,0,0,132,132,0,0,0,0,...,0,0,0,0,0,90,0,90,0,0
9,Outpatient curative care,NaN,38282,0,NaN,18952,7745,0,NaN,437,...,0,0,0,0,0,396,0,396,0,0
24,Medical goods,33584,0,0,1102,1102,0,0,0,0,...,0,0,0,0,0,5492,0,5492,0,0
18,Long-term care (health),57387,1324,37483,14386,197,0,14047,142,0,...,0,0,0,0,0,4194,4194,0,0,0


In [42]:
expenditure_function_clean.to_csv('UK_Accounts_expenditure_by_function_clean.csv', index=False)